# Bronze Layer - Raw Data Ingestion

## Objective

The Bronze Layer is responsible for ingesting raw ride-sharing datasets into Apache Spark without applying any business transformations.

The following datasets are loaded:

- `drivers.csv`
- `trips.csv`
- `trip_logs.csv`

## Bronze Layer Responsibilities

- Define explicit schemas
- Read raw CSV files using PySpark
- Preserve source data without transformation
- Validate dataset availability
- Verify record counts and schemas
- Store raw datasets in Parquet format
- Confirm that source and Bronze record counts match

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    FloatType,
    TimestampType
)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/rideshare_data"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/rideshare_data/drivers.csv,drivers.csv,3618,1783709895000
dbfs:/Volumes/workspace/default/rideshare_data/trip_logs.csv,trip_logs.csv,6036,1783709915000
dbfs:/Volumes/workspace/default/rideshare_data/trips.csv,trips.csv,7049,1783709934000


In [0]:
base_path = "/Volumes/workspace/default/rideshare_data"

drivers_path = f"{base_path}/drivers.csv"
trips_path = f"{base_path}/trips.csv"
trip_logs_path = f"{base_path}/trip_logs.csv"

In [0]:
drivers_schema = StructType([
    StructField("driver_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("rating", FloatType(), True)
])

In [0]:
trips_schema = StructType([
    StructField("trip_id", IntegerType(), True),
    StructField("driver_id", IntegerType(), True),
    StructField("pickup_location", StringType(), True),
    StructField("drop_location", StringType(), True),
    StructField("distance_km", FloatType(), True),
    StructField("fare_amount", FloatType(), True),
    StructField("trip_status", StringType(), True)
])

In [0]:
trip_logs_schema = StructType([
    StructField("log_id", IntegerType(), True),
    StructField("trip_id", IntegerType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("delay_minutes", FloatType(), True),
    StructField("cancellation_flag", IntegerType(), True)
])

In [0]:
drivers_df = (
    spark.read
    .option("header", True)
    .schema(drivers_schema)
    .csv(drivers_path)
)

display(drivers_df)

driver_id,name,city,rating
1,Rahul_1,Delhi,4.6
2,Priya_2,Mumbai,3.7
3,Rahul_3,Pune,3.6
4,Sneha_4,Delhi,3.5
5,Priya_5,Mumbai,4.3
6,Amit_6,Pune,3.8
7,Anjali_7,Hyderabad,3.8
8,Arjun_8,Bangalore,4.7
9,Amit_9,Mumbai,4.5
10,Vikas_10,Bangalore,3.7


In [0]:
trips_df = (
    spark.read
    .option("header", True)
    .schema(trips_schema)
    .csv(trips_path)
)

display(trips_df)

trip_id,driver_id,pickup_location,drop_location,distance_km,fare_amount,trip_status
1,78,IT Park,IT Park,8.6,0.0,Cancelled
2,114,Airport,Mall,17.54,203.08,Completed
3,132,Railway Station,Mall,17.27,213.02,Completed
4,58,Railway Station,IT Park,20.55,185.6,Completed
5,19,IT Park,IT Park,12.47,177.11,Completed
6,103,Airport,IT Park,7.61,95.83,Completed
7,57,Airport,City Center,6.05,0.0,Cancelled
8,64,IT Park,City Center,23.1,204.42,Completed
9,144,City Center,City Center,15.7,0.0,Cancelled
10,110,IT Park,IT Park,21.1,249.72,Completed


In [0]:
trip_logs_df = (
    spark.read
    .option("header", True)
    .schema(trip_logs_schema)
    .csv(trip_logs_path)
)

display(trip_logs_df)

log_id,trip_id,start_time,end_time,delay_minutes,cancellation_flag
1,1,2025-01-03T01:44:00.000Z,null,0.0,1
2,2,2025-01-02T04:34:00.000Z,2025-01-02T04:50:00.000Z,20.0,0
3,3,2025-01-06T22:55:00.000Z,2025-01-06T23:29:00.000Z,18.0,0
4,4,2025-01-07T22:14:00.000Z,2025-01-07T22:47:00.000Z,18.0,0
5,5,2025-01-05T00:15:00.000Z,2025-01-05T01:15:00.000Z,20.0,0
6,6,2025-01-04T22:19:00.000Z,2025-01-04T22:33:00.000Z,14.0,0
7,7,2025-01-02T02:38:00.000Z,null,0.0,1
8,8,2025-01-06T08:45:00.000Z,2025-01-06T09:30:00.000Z,9.0,0
9,9,2025-01-04T11:00:00.000Z,null,0.0,1
10,10,2025-01-01T23:20:00.000Z,2025-01-01T23:47:00.000Z,16.0,0


In [0]:
print("Drivers count:", drivers_df.count())
print("Trips count:", trips_df.count())
print("Trip logs count:", trip_logs_df.count())

Drivers count: 150
Trips count: 150
Trip logs count: 150


In [0]:
drivers_df.printSchema()
trips_df.printSchema()
trip_logs_df.printSchema()

root
 |-- driver_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- rating: float (nullable = true)

root
 |-- trip_id: integer (nullable = true)
 |-- driver_id: integer (nullable = true)
 |-- pickup_location: string (nullable = true)
 |-- drop_location: string (nullable = true)
 |-- distance_km: float (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- trip_status: string (nullable = true)

root
 |-- log_id: integer (nullable = true)
 |-- trip_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- delay_minutes: float (nullable = true)
 |-- cancellation_flag: integer (nullable = true)



In [0]:
bronze_base_path = "/Volumes/workspace/default/rideshare_data/bronze"

drivers_df.write.mode("overwrite").parquet(f"{bronze_base_path}/drivers")
trips_df.write.mode("overwrite").parquet(f"{bronze_base_path}/trips")
trip_logs_df.write.mode("overwrite").parquet(f"{bronze_base_path}/trip_logs")

print("Bronze layer data saved successfully.")

Bronze layer data saved successfully.


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/rideshare_data/bronze"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/rideshare_data/bronze/drivers/,drivers/,0,1783710869215
dbfs:/Volumes/workspace/default/rideshare_data/bronze/trip_logs/,trip_logs/,0,1783710869215
dbfs:/Volumes/workspace/default/rideshare_data/bronze/trips/,trips/,0,1783710869215


In [0]:
bronze_drivers_df = spark.read.parquet(
    "/Volumes/workspace/default/rideshare_data/bronze/drivers"
)

bronze_trips_df = spark.read.parquet(
    "/Volumes/workspace/default/rideshare_data/bronze/trips"
)

bronze_trip_logs_df = spark.read.parquet(
    "/Volumes/workspace/default/rideshare_data/bronze/trip_logs"
)

print("Bronze drivers:", bronze_drivers_df.count())
print("Bronze trips:", bronze_trips_df.count())
print("Bronze trip logs:", bronze_trip_logs_df.count())

Bronze drivers: 150
Bronze trips: 150
Bronze trip logs: 150


In [0]:
bronze_summary = [
    ("drivers", bronze_drivers_df.count()),
    ("trips", bronze_trips_df.count()),
    ("trip_logs", bronze_trip_logs_df.count())
]

bronze_summary_df = spark.createDataFrame(
    bronze_summary,
    ["dataset_name", "record_count"]
)

display(bronze_summary_df)

dataset_name,record_count
drivers,150
trips,150
trip_logs,150


## Ingestion Validation Summary

All three source datasets were successfully ingested into the Bronze Layer.

| Dataset | Source Records | Bronze Records | Status |
|---|---:|---:|---|
| Drivers | 150 | 150 | Passed |
| Trips | 150 | 150 | Passed |
| Trip Logs | 150 | 150 | Passed |

No records were lost during the ingestion process.

## Bronze Layer Conclusion

The Bronze Layer was completed successfully.

- Raw datasets were loaded using explicit PySpark schemas.
- No cleaning or transformation was applied.
- All source records were preserved.
- The datasets were stored in Parquet format.
- Source-to-Bronze record count validation passed.

The data is now ready for cleaning, integration, validation, and enrichment in the Silver Layer.